
df = pd.read_csv("titanic.csv")


## Part 1 — NumPy: arrays vs. lists

In [ ]:
import numpy as np

# A Python list vs a NumPy array holding the same numbers
py_list = [1, 2, 3, 4, 5]
np_array = np.array(py_list)

print(type(py_list), py_list)
print(type(np_array), np_array)
print("dtype:", np_array.dtype)

<class 'list'> [1, 2, 3, 4, 5]
<class 'numpy.ndarray'> [1 2 3 4 5]
dtype: int64


In [ ]:
# Speed comparison squaring 1,000,000 numbers
import time

n = 1_000_000
big_list = list(range(n))
big_array = np.arange(n)

start = time.time()
squared_list = [x ** 2 for x in big_list]          # Python loop
list_time = time.time() - start

start = time.time()
squared_array = big_array ** 2                      # vectorized NumPy op
array_time = time.time() - start

print(f"List comprehension: {list_time:.4f}s")
print(f"NumPy vectorized:   {array_time:.4f}s")
print(f"NumPy was ~{list_time / array_time:.1f}x faster")

List comprehension: 0.0956s
NumPy vectorized:   0.0032s
NumPy was ~29.4x faster


### Array creation

In [ ]:
# Common ways to create arrays
zeros = np.zeros((2, 3))          # 2x3 array of zeros
ones = np.ones((3,))              # 1D array of ones
range_arr = np.arange(0, 10, 2)   # start, stop, step -> [0 2 4 6 8]
lin_arr = np.linspace(0, 1, 5)    # 5 evenly spaced points between 0 and 1
rand_arr = np.random.default_rng(42).integers(0, 100, size=(3, 3))  # random 3x3 ints

print("zeros:\n", zeros)
print("ones:", ones)
print("arange:", range_arr)
print("linspace:", lin_arr)
print("random ints:\n", rand_arr)
print("shape:", rand_arr.shape, "| ndim:", rand_arr.ndim, "| size:", rand_arr.size)

zeros:
 [[0. 0. 0.]
 [0. 0. 0.]]
ones: [1. 1. 1.]
arange: [0 2 4 6 8]
linspace: [0.   0.25 0.5  0.75 1.  ]
random ints:
 [[ 8 77 65]
 [43 43 85]
 [ 8 69 20]]
shape: (3, 3) | ndim: 2 | size: 9


### Indexing & slicing

In [ ]:
matrix = np.arange(1, 13).reshape(3, 4)
print("matrix:\n", matrix)

print("single element [1,2]:", matrix[1, 2])
print("first row:", matrix[0, :])
print("last column:", matrix[:, -1])
print("sub-block rows 0-1, cols 1-2:\n", matrix[0:2, 1:3])

# Boolean indexing: pick elements matching a condition
mask = matrix % 2 == 0
print("even mask:\n", mask)
print("even values:", matrix[mask])

matrix:
 [[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]
single element [1,2]: 7
first row: [1 2 3 4]
last column: [ 4  8 12]
sub-block rows 0-1, cols 1-2:
 [[2 3]
 [6 7]]
even mask:
 [[False  True False  True]
 [False  True False  True]
 [False  True False  True]]
even values: [ 2  4  6  8 10 12]


### Broadcasting



In [5]:
row = np.array([1, 2, 3, 4])
col = np.array([[10], [20], [30]])

# row (1x4) broadcasts against col (3x1) -> result is (3x4)
result = row + col
print("row:", row)
print("col:\n", col)
print("row + col (broadcast):\n", result)

# Common real use: normalize each column of a matrix by its mean
data = np.array([[1., 2., 3.], [4., 5., 6.], [7., 8., 9.]])
col_means = data.mean(axis=0)          # shape (3,)
normalized = data - col_means          # broadcasts (3,3) - (3,) -> (3,3)
print("column means:", col_means)
print("normalized:\n", normalized)

row: [1 2 3 4]
col:
 [[10]
 [20]
 [30]]
row + col (broadcast):
 [[11 12 13 14]
 [21 22 23 24]
 [31 32 33 34]]
column means: [4. 5. 6.]
normalized:
 [[-3. -3. -3.]
 [ 0.  0.  0.]
 [ 3.  3.  3.]]


### Vectorized operations

In [6]:
a = np.array([1, 2, 3, 4, 5])
b = np.array([10, 20, 30, 40, 50])

print("a + b        =", a + b)
print("a * b        =", a * b)
print("b / a        =", b / a)
print("a ** 2       =", a ** 2)
print("sum(a)       =", a.sum())
print("mean(b)      =", b.mean())
print("a > 2        =", a > 2)          # elementwise comparison
print("np.where     =", np.where(a > 2, "big", "small"))  # vectorized if/else

a + b        = [11 22 33 44 55]
a * b        = [ 10  40  90 160 250]
b / a        = [10. 10. 10. 10. 10.]
a ** 2       = [ 1  4  9 16 25]
sum(a)       = 15
mean(b)      = 30.0
a > 2        = [False False  True  True  True]
np.where     = ['small' 'small' 'big' 'big' 'big']


## Part 2 — Pandas: load, explore, clean, transform

### Load raw data



In [7]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(7)

n = 30
names = [f"Passenger_{i}" for i in range(1, n + 1)]
sexes = rng.choice(["male", "female", "Male", "FEMALE"], size=n)   # inconsistent casing on purpose
pclass = rng.choice([1, 2, 3], size=n, p=[0.2, 0.3, 0.5])
age = rng.normal(30, 12, size=n).round(1)
fare = rng.normal(35, 20, size=n).round(2)
survived = rng.choice([0, 1], size=n)
embarked = rng.choice(["S", "C", "Q", None], size=n, p=[0.5, 0.25, 0.2, 0.05])

df = pd.DataFrame({
    "PassengerId": range(1, n + 1),
    "Name": names,
    "Sex": sexes,
    "Pclass": pclass,
    "Age": age,
    "Fare": fare,
    "Survived": survived,
    "Embarked": embarked,
})

# Inject realistic messiness: missing ages, negative fare typo, duplicate rows
df.loc[rng.choice(n, 5, replace=False), "Age"] = np.nan
df.loc[2, "Fare"] = -5.0                     # bad data: negative fare
df = pd.concat([df, df.iloc[[3, 10]]], ignore_index=True)  # duplicate a couple of rows

df.head()

,PassengerId,Name,Sex,Pclass,Age,Fare,Survived,Embarked
0,1,Passenger_1,FEMALE,3,46.3,60.14,1,S
1,2,Passenger_2,Male,3,11.4,48.79,0,C
2,3,Passenger_3,Male,3,40.3,-5.00,0,Q
3,4,Passenger_4,FEMALE,3,31.4,27.63,1,C
4,5,Passenger_5,Male,3,22.3,30.00,0,C


### Explore the structure

In [8]:
print(df.shape)
df.info()

(32, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  32 non-null     int64  
 1   Name         32 non-null     object 
 2   Sex          32 non-null     object 
 3   Pclass       32 non-null     int64  
 4   Age          27 non-null     float64
 5   Fare         32 non-null     float64
 6   Survived     32 non-null     int64  
 7   Embarked     29 non-null     object 
dtypes: float64(2), int64(3), object(3)
memory usage: 2.1+ KB


In [9]:
df.describe(include="all")

,PassengerId,Name,Sex,Pclass,Age,Fare,Survived,Embarked
count,32.00000,32,32,32.000000,27.000000,32.000000,32.000000,29
unique,NaN,30,4,NaN,NaN,NaN,NaN,3
top,NaN,Passenger_4,FEMALE,NaN,NaN,NaN,NaN,S
freq,NaN,2,11,NaN,NaN,NaN,NaN,13
mean,15.00000,NaN,NaN,2.343750,29.559259,33.943125,0.687500,NaN
std,8.78415,NaN,NaN,0.827331,11.038195,19.376238,0.470929,NaN
min,1.00000,NaN,NaN,1.000000,6.100000,-5.710000,0.000000,NaN
25%,7.75000,NaN,NaN,2.000000,23.350000,27.332500,0.000000,NaN
50%,14.50000,NaN,NaN,3.000000,29.200000,31.815000,1.000000,NaN
75%,22.25000,NaN,NaN,3.000000,37.350000,47.017500,1.000000,NaN


### Handling missing values

In [10]:
# Where are the gaps?
print(df.isna().sum())

# Strategy: fill missing Age with the median (robust to outliers), and missing
# Embarked with the most frequent port, since dropping rows would lose data.
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

print(df.isna().sum())

PassengerId    0
Name           0
Sex            0
Pclass         0
Age            5
Fare           0
Survived       0
Embarked       3
dtype: int64
PassengerId    0
Name           0
Sex            0
Pclass         0
Age            0
Fare           0
Survived       0
Embarked       0
dtype: int64


### Removing duplicates

In [11]:
print("Before:", df.shape)
df = df.drop_duplicates(subset="PassengerId")
print("After: ", df.shape)

Before: (32, 8)
After:  (30, 8)


### Fixing inconsistent values & bad data

In [12]:
# Standardize casing so 'male'/'Male' aren't treated as different categories
df["Sex"] = df["Sex"].str.lower()

# A fare can't be negative — treat it as missing, then fill with the median
df.loc[df["Fare"] < 0, "Fare"] = np.nan
df["Fare"] = df["Fare"].fillna(df["Fare"].median())

df[["Sex", "Fare"]].describe(include="all")

,Sex,Fare
count,30,30.000000
unique,2,NaN
top,female,NaN
freq,20,NaN
mean,NaN,36.916667
std,NaN,16.853487
min,NaN,1.240000
25%,NaN,28.377500
50%,NaN,34.645000
75%,NaN,47.712500


### Filtering rows

In [13]:
# Rows for female passengers in 1st or 2nd class
subset = df[(df["Sex"] == "female") & (df["Pclass"].isin([1, 2]))]
subset.head()

,PassengerId,Name,Sex,Pclass,Age,Fare,Survived,Embarked
5,6,Passenger_6,female,2,54.0,65.47,1,S
6,7,Passenger_7,female,1,29.2,26.44,1,S
9,10,Passenger_10,female,1,36.9,32.58,1,S
11,12,Passenger_12,female,2,38.2,12.72,1,C
15,16,Passenger_16,female,2,21.9,48.06,0,S


### Groupby

In [14]:
# Survival rate and average fare per class
summary = df.groupby("Pclass").agg(
    survival_rate=("Survived", "mean"),
    avg_fare=("Fare", "mean"),
    count=("PassengerId", "count"),
)
summary

,survival_rate,avg_fare,count
Pclass,,,
1,0.857143,26.865714,7
2,0.571429,39.567857,7
3,0.625000,40.154063,16


### Merging

In [15]:
# A small lookup table, e.g. port name for each embarkation code
port_names = pd.DataFrame({
    "Embarked": ["S", "C", "Q"],
    "PortName": ["Southampton", "Cherbourg", "Queenstown"],
})

df = df.merge(port_names, on="Embarked", how="left")
df[["PassengerId", "Embarked", "PortName"]].head()

,PassengerId,Embarked,PortName
0,1,S,Southampton
1,2,C,Cherbourg
2,3,Q,Queenstown
3,4,C,Cherbourg
4,5,C,Cherbourg


## Part 3 — Full raw-to-clean pipeline

The cells above were run step by step to show *why* each step matters. Here is the same pipeline collected into one script, so you can see the whole transformation from raw to analysis-ready data in one place.

In [16]:
def clean_dataset(raw_df: pd.DataFrame) -> pd.DataFrame:
    """Turn a raw, messy dataframe into an analysis-ready one."""
    out = raw_df.copy()

    # 1. Drop exact duplicate passengers
    out = out.drop_duplicates(subset="PassengerId")

    # 2. Standardize text formatting
    out["Sex"] = out["Sex"].str.lower()

    # 3. Fix impossible values (negative fares) by treating them as missing
    out.loc[out["Fare"] < 0, "Fare"] = np.nan

    # 4. Fill missing numeric values with the median (robust to outliers)
    out["Age"] = out["Age"].fillna(out["Age"].median())
    out["Fare"] = out["Fare"].fillna(out["Fare"].median())

    # 5. Fill missing categorical values with the most frequent category
    out["Embarked"] = out["Embarked"].fillna(out["Embarked"].mode()[0])

    # 6. Enforce sensible dtypes
    out["Pclass"] = out["Pclass"].astype("category")
    out["Survived"] = out["Survived"].astype("int8")

    return out.reset_index(drop=True)


clean_df = clean_dataset(df)
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   PassengerId  30 non-null     int64   
 1   Name         30 non-null     object  
 2   Sex          30 non-null     object  
 3   Pclass       30 non-null     category
 4   Age          30 non-null     float64 
 5   Fare         30 non-null     float64 
 6   Survived     30 non-null     int8    
 7   Embarked     30 non-null     object  
 8   PortName     30 non-null     object  
dtypes: category(1), float64(2), int64(1), int8(1), object(4)
memory usage: 2.0+ KB


### Raw vs. clean, side by side

In [17]:
print("RAW shape:  ", df.shape)
print("CLEAN shape:", clean_df.shape)
print()
print("Missing values remaining:")
print(clean_df.isna().sum())

RAW shape:   (30, 9)
CLEAN shape: (30, 9)

Missing values remaining:
PassengerId    0
Name           0
Sex            0
Pclass         0
Age            0
Fare           0
Survived       0
Embarked       0
PortName       0
dtype: int64


In [18]:
clean_df.head(10)

,PassengerId,Name,Sex,Pclass,Age,Fare,Survived,Embarked,PortName
0,1,Passenger_1,female,3,46.3,60.140,1,S,Southampton
1,2,Passenger_2,male,3,11.4,48.790,0,C,Cherbourg
2,3,Passenger_3,male,3,40.3,34.645,0,Q,Queenstown
3,4,Passenger_4,female,3,31.4,27.630,1,C,Cherbourg
4,5,Passenger_5,male,3,22.3,30.000,0,C,Cherbourg
5,6,Passenger_6,female,2,54.0,65.470,1,S,Southampton
6,7,Passenger_7,female,1,29.2,26.440,1,S,Southampton
7,8,Passenger_8,male,3,15.6,28.930,0,S,Southampton
8,9,Passenger_9,male,1,30.9,42.050,1,S,Southampton
9,10,Passenger_10,female,1,36.9,32.580,1,S,Southampton
